# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente inicializado con éxito')

Cliente inicializado con éxito


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{'role': 'user','content': prompt_zero_shot}]
)

print('Zero-shot: ', response_zero.choices[0].message.content)

Zero-shot:  Mixto.


In [ ]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{'role': 'user','content': prompt_few_shot}]
)

print('Few-shot: ', response_few.choices[0].message.content)

Few-shot:  Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{'role': 'user','content': problema}]
)

print('cot-shot: ', response_cot.choices[0].message.content)

cot-shot:  **Paso a paso**

1. **Velocidades de los trenes**  
   - Tren 1 (primer tren): \(v_1 = 80\ \text{km/h}\)  
   - Tren 2 (segundo tren): \(v_2 = 120\ \text{km/h}\)

2. **Tiempo de ventaja**  
   El primer tren parte 2 h antes que el segundo.

3. **Distancia que el primer tren recorre en esas 2 h**  
   \[
   d_{\text{ventaja}} = v_1 \times \Delta t = 80\ \text{km/h} \times 2\ \text{h} = 160\ \text{km}
   \]
   Por lo tanto, cuando el segundo tren empieza su marcha, el primero ya está 160 km adelante.

4. **Velocidad relativa**  
   La diferencia de velocidades determina cuán rápido se reduce la distancia:
   \[
   v_{\text{rel}} = v_2 - v_1 = 120\ \text{km/h} - 80\ \text{km/h} = 40\ \text{km/h}
   \]

5. **Tiempo que tarda el segundo tren en alcanzar al primero**  
   \[
   t_{\text{coger}} = \frac{d_{\text{ventaja}}}{v_{\text{rel}}} = \frac{160\ \text{km}}{40\ \text{km/h}} = 4\ \text{h}
   \]

   Así, el segundo tren necesita **4 horas** desde su salida para alcanzar al prime

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{'role': 'user','content': prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, pero no puedo ayudar con esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print('Embeddings generados:', embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = '¿Puedo devolver algo que compré en oferta?'
fragmento = buscar_fragmento(pregunta)
print('Fragmento recuperado:', fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

No, los productos en oferta no son elegibles para devolución, solo para cambio de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [4]:
# Leer API key, instalar e importar librerías

!pip install -q sentence-transformers
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [5]:
import pprint
from sentence_transformers import SentenceTransformer
import numpy as np

from google.colab import userdata
from groq import Groq

class LlmComm:
    def __init__(self, api_key: str = userdata.get('GROQ_API_KEY'), role: str = 'user', model: str = 'openai/gpt-oss-20b'):
        try:
            self.client = Groq(api_key=userdata.get('GROQ_API_KEY'))
            self.role = role
            self.model = model
            self.response = None
            self.prompt = ''
            print('Conexión creada con éxito')
        except Exception as e:
            print(e)

    def preguntar(self, prompt: str):
        self.prompt = prompt
        try:
            self.response = self.client.chat.completions.create(
                model=self.model,
                messages=[{'role': self.role, 'content': self.prompt}]
            )
            return self.response.choices[0].message.content
        except Exception as e:
            print(e)

    def obtener_metricas_pregunta(self):
        if self.response:
            usage = {
                'prompt': self.prompt,
                'completion_tokens': self.response.usage.completion_tokens,
                'prompt_tokens': self.response.usage.prompt_tokens,
                'total_tokens': self.response.usage.total_tokens,
                'completion_time': self.response.usage.completion_time,
                'completion_tokens_details': self.response.usage.completion_tokens_details,
                'prompt_time': self.response.usage.prompt_time,
                'prompt_tokens_details': self.response.usage.prompt_tokens_details,
                'queue_time': self.response.usage.queue_time,
                'total_time': self.response.usage.total_time
            }
            return usage

    def obtener_texto_respuesta(self):
        if self.response:
            return self.response.choices[0].message.content

my_llm = LlmComm()
prompt = "Hola"
print(my_llm.preguntar(prompt))
pprint.pp(my_llm.obtener_metricas_pregunta())

Conexión creada con éxito
¡Hola! ¿En qué puedo ayudarte hoy?
{'prompt': 'Hola',
 'completion_tokens': 42,
 'prompt_tokens': 72,
 'total_tokens': 114,
 'completion_time': 0.043237841,
 'completion_tokens_details': CompletionTokensDetails(reasoning_tokens=23),
 'prompt_time': 0.003421194,
 'prompt_tokens_details': None,
 'queue_time': 0.086083915,
 'total_time': 0.046659035}


In [14]:
# Definir la lista documentos y generar sus embeddings

class Embedder:
    def __init__(self, corpus, model='paraphrase-multilingual-MiniLM-L12-v2'):
        try:
            self.corpus = corpus
            self.model = model

            self.st = SentenceTransformer(self.model)
            self.embeddings_corpus = self.st.encode(self.corpus)
            self.embeddings_corpus = self.embeddings_corpus / np.linalg.norm(self.embeddings_corpus, axis=1, keepdims=True)

            print('Embeddings generados:', self.embeddings_corpus.shape)

        except Exception as e:
            print(e)

    def buscar_fragmento(self, pregunta):
        embedding_pregunta = self.st.encode([pregunta])
        embedding_pregunta = embedding_pregunta / np.linalg.norm(embedding_pregunta, axis=1, keepdims=True)

        similitudes = np.dot(self.embeddings_corpus, embedding_pregunta.T).flatten()

        indice_mas_similar = np.argmax(similitudes)

        return self.corpus[indice_mas_similar]

corpus = [
    "\\begin{center}\n    \\textbf{\\huge Analysis of the Rosenblatt Perceptron: Convergence and Practical Application} \\\\\n    \\vspace{0.5cm}\n    \\textit{From the mathematical proof of its convergence to a Python implementation} \\\\\n    \\vspace{1cm}\n    \\textbf{Hernández Pérez Erick Fernando} \\\\\n    \\vspace{0.5cm}\n\\end{center}\n\n\\begin{figure}[h!]\n    \\centering\n    \\includegraphics[width=0.5\\textwidth]{img/perceptron.png} \n    \\caption{Oversimplified icon of a perceptron}\n    \\label{fig:imagen}\n\\end{figure}\n\n\\noindent A world where machines can learn by themselves, adapt to new situations and make decisions without having to be constantly reprogrammed. This idea, which is now part of our reality, was for centuries an unattainable dream. From the first attempts to create automata in ancient times to the sophisticated computers of the 20th century, humans have always sought ways to replicate their own intelligence in inert matter. However, although machines could process information at unimaginable speeds, they remained rigid, unable to learn from experience as the human brain does.\n\nIt was in this context of curiosity and intellectual ambition that the first attempts to model thought arose, inspired by the functioning of neurons. The question was challenging: could an artificial structure learn and make decisions as living beings do? The answer to this question would mark a before and after in the field of artificial intelligence, laying the foundations for what we now know as machine learning.\n\nThe road was long, full of skepticism and obstacles, but one discovery in particular would open the door to a new era in computing and forever change the way we understand machine learning.\n\nAs Haykin mentions, the perceptron holds a special place in the historical development of neural networks: it was the first neural network described algorithmically. Its invention by Rosenblatt, a psychologist, inspired engineers, physicists, and mathematicians alike to devote their research efforts to different aspects of neural networks in the 1960s and 1970s (2009)\\cite{haykin}. It was in 1958 that Rosenblatt proposed the perceptron as the first model for learning with a teacher, what is often called supervised learning. If you want to delve deeper into the McCulloch Pitts model, you can see the third issue of this series. \\href{https://github.com/Erick-FHP/Series/blob/main/3_Implementing\n\nA perceptron is an artificial neuron, and therefore a unit of a neural network. This is a different model than the one proposed by McCulloch-Pitts, although it served as the basis for the model proposed by Frank Rosenblatt. However, this term is also often used to refer to the algorithm for supervised learning of binary classifiers. This algorithm is responsible for ensuring that artificial neurons learn and, therefore, can handle the elements of a series of data for classification. \n\nIt is important to emphasize the “neural network unit” part. A single perceptron is capable of classifying data by itself, but it has a very important limitation: linearly non-separable data sets. This is the case of the XOR function that was discussed in the McCulloch-Pitts entry.\n\nThe goal of the perceptron is to correctly classify a sequence of inputs $x_1, x_2, ..., x_n$ into two classes $C_1$ and $C_2$. To do this, the perceptron will return for each point a value $\\hat{y}$ which will be 1 for $C_1$ and -1 for $C_2$.",
    "Since our perceptron assumes that the data is linearly separable, this means that:\n\n$$\\exists \\textbf{w*}: \\forall (x_i,y_i)\\in D\\hspace{1cm} y_i{{\\textbf{w*}}}^T\\textbf{x}_i>0$$\n\nIt is also important to note that by finding a $\\textbf{w*}$ we can find another one, simply by rescaling it $\\textbf{w*}'=\\alpha \\textbf{w*}$. This way we will choose our $\\textbf{w*}$ such that $||\\textbf{w*}||=1$.\n\nAfter this, we are going to rescale our data so that it falls on a circle of radius 1. To do this we do $\\forall i: ||x_i|| \\leq 1$ by dividing each $x_i$ by the maximum norm of the points. This will help us with the proof. Also note that scaling will not affect our solution at all, because once it is found, we can return to the original scale.\n\nWe also define what our margin is. It is the smallest distance between our data and our decision region:\n$$\\gamma = \\text{min}_{x_i,y_i \\in D} |\\textbf{x}^T\\textbf{w*}|>0$$\n\nWe will assume that the value of $\\eta = 1$ and that our weights are $\\textbf{w} = 0$\n\nNow we can start with the demonstration. First we will see how $w^Tw*$ changes in an update:\n\n$$\n\\textbf{w}^T\\textbf{w*} \\rightarrow (\\textbf{w}+y\\textbf{x})^T\\textbf{w*} = \\textbf{w}^T\\textbf{w*}+y\\textbf{x}^T\\textbf{w*} = \\textbf{w}^T\\textbf{w*}+y\\textbf{w*}^T\\textbf{x}\n$$\n\nWhere the term $y\\textbf{w*}^T\\textbf{x}$ by our assumption is positive. This means that our initial product becomes increasingly positive and since we have established our $\\gamma$ we can conclude that:\n\n$$\n(\\textbf{w}+y\\textbf{x})^T\\textbf{w*} \\geq \\textbf{w}^T\\textbf{w*} + \\gamma\n$$\n\nThis means that if we make an update, our inner product of the hyperplane we are searching for with the hyperplane we want to find grows by at least $\\gamma$. This is our lower bound.\n\nAhora checaremos como cambia $\\textbf{w}^T\\textbf{w}$.\n\n$$\n\\textbf{w}^T\\textbf{w}\\rightarrow (\\textbf{w}+y\\textbf{x})^T(\\textbf{w}+y\\textbf{x}) = \\textbf{w}^T\\textbf{w}+2y\\textbf{w}^T\\textbf{x}+y^2\\textbf{x}^T\\textbf{x}\n$$\n\nSince we are doing an update, that means that $y\\textbf{w}^T\\textbf{x}<0$; we also have that $y^2 = 1$ and $\\textbf{x}^T\\textbf{x}\\leq1$. Therefore, we can conclude that:\n\n$$\n(\\textbf{w}+y\\textbf{x})^T(\\textbf{w}+y\\textbf{x}) \\leq \\textbf{w}^T\\textbf{w}+1\n$$\n\nThis being our upper bound.\n\nNow consider the case that occurs after M updates, and applying the Cauchy-Schwarz inequality:\n\n$$M\\gamma \\leq \\textbf{w}^T\\textbf{w*}=|\\textbf{w}^T\\textbf{w*}|\\leq ||\\textbf{w}||\\cdot ||\\textbf{w*}|| = ||\\textbf{w}||=\\sqrt{\\textbf{w}^T\\textbf{w}}\\leq \\sqrt{M}$$\n\nAnd this can be expressed as:\n\n$$M\\gamma \\leq \\sqrt{M} \\rightarrow \\sqrt{M}\\gamma \\leq 1 \\rightarrow M \\leq \\frac{1}{\\gamma^2}$$\n\nWhich means that our number of iterations cannot be greater than the inverse of the square of our margin.",
    "\\begin{itemize}\n    \\item \\textbf{Inputs}: Inputs are the values that enter the perceptron for its respective calculation.\n    \\item \\textbf{Weights}: Weights are numerical values that determine the importance of each input. Each input is multiplied by its corresponding weight. The weights are adjusted during the learning process to improve the accuracy of the perceptron.\n    \\item \\textbf{Bias}, which is also often referred to as a \\textbf{threshold value}: The bias is a constant value (often 1) that is added to the weighted sum of the inputs and the weights. Like the weights, the bias value is adjusted during the learning process.\n    \\item \\textbf{Weighted sum}: The weighted sum is the result of multiplying each input by its corresponding weight and then adding all these products. To this we add the bias. This sum is called the induced local field. Mathematically, if we have n inputs $x_1,x_2,…,x_n$ and their corresponding weights $w_1,w_2,…,w_n$ and the bias b, the weighted sum is calculated as follows:\n    $$\n    \\sum_{i=1}^{n} w_ix_i + b\n    $$\n    Another way to deal with the bias is to consider it as a synaptic weight with a fixed input of +1, this gives us a more compact way to work:\n    $$\n    \\sum_{i=0}^{n}w_ix_i=w_0x_0+\\sum_{i=1}^n w_ix_i = b + \\sum_{i=1}^n w_ix_i\n    $$\n    \\item \\textbf{Activation Function}: The activation function $\\phi$ takes the weighted sum and produces the final output of the perceptron. In our case, our activation function is the sign function.\n    \n    $$\n    \\phi = \\text{sgn}(x) = \n    \\begin{cases}\n    1 & \\text{si } x > 0, \\\\\n    -1 & \\text{si } x < 0.\n    \\end{cases}\n    $$\n\\end{itemize}\n\nThe total entries can be grouped into a single vector represented by $\\mathbf{x} \\in \\mathbb{R}^{n+1 \\times 1} = (+1, x_1, x_2,...,x_n)^T$. Similarly, all the weights can be put together in a vector that is represented by $\\mathbf{w}\\in \\mathbb{R}^{n+1 \\times 1} = (b, w_1, w_2,...,w_n)^T$\n\nThus, using linear algebra, we can write the perceptron calculation as follows:\n\n$$\n\\hat{y} = \\phi(\\textbf{w}^T\\textbf{x})\n$$"
]

my_embedder = Embedder(corpus)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [13]:
# Definir la función buscar_fragmento
# NOTA: Esta definida en la clase, aún así, la coloco aquí (aunque usaré la de mi clase)

def buscar_fragmento(pregunta, st, embeddings_corpus, corpus):
    embedding_pregunta = st.encode([pregunta])
    embedding_pregunta = embedding_pregunta / np.linalg.norm(embedding_pregunta, axis=1, keepdims=True)
    similitudes = np.dot(embeddings_corpus, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return corpus[indice_mas_similar]

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [23]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
prompt = "¿Cómo crece el producto punto respecto al hiperplano que estamos buscando respecto al hiperplano que queremos encontrar?"
respuesta_sin_rag = my_llm.preguntar(prompt)
print(respuesta_sin_rag)

**Respuesta breve (en español)**  

En la teoría de los hiper‑planos (por ejemplo, en SVM o en cualquier clasificador lineal) el **producto punto** \(w\!\cdot\!x\) es una función *lineal* de los parámetros del hiper‑plano (\(w\) y \(b\)).  
* Si mantenemos \(w\) fijo y cambiamos el punto \(x\), el valor de \(w\!\cdot\!x\) crece (o decrece) exactamente en la proyección de \(x\) sobre el vector normal \(w\).  
* Si mantenemos los puntos fijos y cambiamos el vector normal \(w\), el producto punto crece (o decrece) a razón de \(\nabla_w (w\!\cdot\!x)=x\); es decir, en la dirección del propio vector \(x\).  
* Cuando **escala** el vector normal (\(w'=\lambda w\)), el producto punto escala también: \(w'\!\cdot\!x=\lambda (w\!\cdot\!x)\).  
* La distancia de un punto \(x\) al hiper‑plano \(w\!\cdot\!x + b = 0\) es \(\displaystyle \frac{|w\!\cdot\!x + b|}{\|w\|}\). Por tanto, el **producto punto** aumenta linealmente con la distancia al hiper‑plano (cuando el signo es correcto).  

En pocas pa

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [29]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
pregunta = "¿Cómo crece el producto punto respecto al hiperplano que estamos buscando respecto al hiperplano que queremos encontrar?"

contexto = my_embedder.buscar_fragmento(pregunta)

prompt = f"""Responde la pregunta utilizando únicamente la información proporcionada en el contexto.

Contexto:
{contexto}

Pregunta:
{pregunta}

Respuesta:"""
print(contexto)
print('='*100)
respuesta_con_rag = my_llm.preguntar(prompt)
print(respuesta_con_rag)

Since our perceptron assumes that the data is linearly separable, this means that:

$$\exists \textbf{w*}: \forall (x_i,y_i)\in D\hspace{1cm} y_i{{\textbf{w*}}}^T\textbf{x}_i>0$$

It is also important to note that by finding a $\textbf{w*}$ we can find another one, simply by rescaling it $\textbf{w*}'=\alpha \textbf{w*}$. This way we will choose our $\textbf{w*}$ such that $||\textbf{w*}||=1$.

After this, we are going to rescale our data so that it falls on a circle of radius 1. To do this we do $\forall i: ||x_i|| \leq 1$ by dividing each $x_i$ by the maximum norm of the points. This will help us with the proof. Also note that scaling will not affect our solution at all, because once it is found, we can return to the original scale.

We also define what our margin is. It is the smallest distance between our data and our decision region:
$$\gamma = \text{min}_{x_i,y_i \in D} |\textbf{x}^T\textbf{w*}|>0$$

We will assume that the value of $\eta = 1$ and that our weights are $\textbf{w}

In [25]:
# Una extra
pregunta = "¿Qué componentes forman un perceptrón y cuál es la función de activación utilizada?"

contexto = my_embedder.buscar_fragmento(pregunta)

prompt = f"""Responde la pregunta utilizando únicamente la información proporcionada en el contexto.

Contexto:
{contexto}

Pregunta:
{pregunta}

Respuesta:"""
print(contexto)
print('='*100)
print(my_llm.preguntar(prompt))

\begin{itemize}
    \item \textbf{Inputs}: Inputs are the values that enter the perceptron for its respective calculation.
    \item \textbf{Weights}: Weights are numerical values that determine the importance of each input. Each input is multiplied by its corresponding weight. The weights are adjusted during the learning process to improve the accuracy of the perceptron.
    \item \textbf{Bias}, which is also often referred to as a \textbf{threshold value}: The bias is a constant value (often 1) that is added to the weighted sum of the inputs and the weights. Like the weights, the bias value is adjusted during the learning process.
    \item \textbf{Weighted sum}: The weighted sum is the result of multiplying each input by its corresponding weight and then adding all these products. To this we add the bias. This sum is called the induced local field. Mathematically, if we have n inputs $x_1,x_2,…,x_n$ and their corresponding weights $w_1,w_2,…,w_n$ and the bias b, the weighted sum is 

**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [30]:
# Mostrar ambas respuestas para comparar
print(f'Respuesta sin rag: \n{respuesta_sin_rag}')
print('')
print('=+='*40)
print('')
print(f'Respuesta con rag: \n{respuesta_con_rag}')

Respuesta sin rag: 
**Respuesta breve (en español)**  

En la teoría de los hiper‑planos (por ejemplo, en SVM o en cualquier clasificador lineal) el **producto punto** \(w\!\cdot\!x\) es una función *lineal* de los parámetros del hiper‑plano (\(w\) y \(b\)).  
* Si mantenemos \(w\) fijo y cambiamos el punto \(x\), el valor de \(w\!\cdot\!x\) crece (o decrece) exactamente en la proyección de \(x\) sobre el vector normal \(w\).  
* Si mantenemos los puntos fijos y cambiamos el vector normal \(w\), el producto punto crece (o decrece) a razón de \(\nabla_w (w\!\cdot\!x)=x\); es decir, en la dirección del propio vector \(x\).  
* Cuando **escala** el vector normal (\(w'=\lambda w\)), el producto punto escala también: \(w'\!\cdot\!x=\lambda (w\!\cdot\!x)\).  
* La distancia de un punto \(x\) al hiper‑plano \(w\!\cdot\!x + b = 0\) es \(\displaystyle \frac{|w\!\cdot\!x + b|}{\|w\|}\). Por tanto, el **producto punto** aumenta linealmente con la distancia al hiper‑plano (cuando el signo es corre

Considerando las dos respuestas, aunque ambas con correctas, podemos notar que la `sin_rag` tiene a debrayarse pues no tiene en que basarse para la respuesta. En cambio, la `con_rag` es más compacta y va directo al grano respecto al `corpus`